# Drawing pixel baseline

Run cells from top to bottom. This notebook is the interactive entry point; reusable implementation and tested data/model logic remain in `src/train_drawing_models.py`. Set output paths before running the training cell to preserve earlier artifacts.

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
assert (PROJECT_ROOT / 'src').is_dir(), 'Open this notebook from the repository or notebooks directory'
sys.path.insert(0, str(PROJECT_ROOT))

from src.train_drawing_models import DEFAULT_IMAGE_ROOT, build_manifest, run_experiment

IMAGE_ROOT = DEFAULT_IMAGE_ROOT
MODEL_PATH = PROJECT_ROOT / 'models' / 'finalization_baseline' / 'drawing_image_model.joblib'
RESULTS_DIR = PROJECT_ROOT / 'results' / 'finalization_baseline' / 'drawing'
SEED = 42
IMAGE_SIZE = 32
print('Image root:', IMAGE_ROOT)
print('Image root exists:', IMAGE_ROOT.exists())

Image root: C:\Users\daisy\Projects\AI\pd-ml-lab\data\Parkinson's drawings dataset
Image root exists: True


In [2]:
# Inspect discovered files, task/class counts, and provisional filename groups before training.
manifest = build_manifest(IMAGE_ROOT)
display(manifest.groupby(['task', 'label']).size().rename('images').to_frame())
display(manifest[['path', 'task', 'label', 'group']].head())
print('Rows:', len(manifest), '| candidate groups:', manifest['group'].nunique())

images
task   label            
spiral healthy        51
       parkinson      51
wave   healthy        51
       parkinson      51

,path,task,label,group
0,C:\Users\daisy\Projects\AI\pd-ml-lab\data\Park...,spiral,healthy,healthy:V01
1,C:\Users\daisy\Projects\AI\pd-ml-lab\data\Park...,spiral,healthy,healthy:V01
2,C:\Users\daisy\Projects\AI\pd-ml-lab\data\Park...,spiral,healthy,healthy:V02
3,C:\Users\daisy\Projects\AI\pd-ml-lab\data\Park...,spiral,healthy,healthy:V02
4,C:\Users\daisy\Projects\AI\pd-ml-lab\data\Park...,spiral,healthy,healthy:V03


Rows: 204 | candidate groups: 28


In [3]:
# Run the full train/CV/selection/held-out evaluation and write the model and reports.
summary = run_experiment(
    image_root=IMAGE_ROOT,
    model_path=MODEL_PATH,
    results_dir=RESULTS_DIR,
    seed=SEED,
    image_size=IMAGE_SIZE,
)
summary

Grouped split summary:
            images  participant_groups  healthy_images  parkinson_images  healthy_groups  parkinson_groups
train          114                  19              54                60               9                10
validation      42                   5              18                24               3                 2
test            48                   4              30                18               1                 3

Majority baseline CV ROC-AUC: 0.500 +/- 0.000
Candidate model CV and validation results:
              model  cv_train_roc_auc_mean  cv_roc_auc_mean  cv_roc_auc_std  cv_roc_auc_gap  cv_train_balanced_accuracy_mean  cv_balanced_accuracy_mean  validation_roc_auc  validation_balanced_accuracy  validation_sensitivity  validation_specificity  validation_precision  validation_f1  validation_false_positives  validation_false_negatives validation_confusion_matrix                                                             best_params
logistic_regress

{'study_scope': "Exploratory healthy-versus-known-Parkinson's image classification; not prospective early-risk prediction",
 'seed': 42,
 'image_size': 32,
 'label_mapping': {'healthy': 0, 'parkinson': 1},
 'group_key': 'class-scoped filename prefix V## (provisional participant token; verify with source documentation)',
 'input_root': "C:\\Users\\daisy\\Projects\\AI\\pd-ml-lab\\data\\Parkinson's drawings dataset",
 'n_images_after_exact_deduplication': 204,
 'n_unique_candidate_groups': 28,
 'split_summary': {'train': {'images': 114,
   'participant_groups': 19,
   'healthy_images': 54,
   'parkinson_images': 60,
   'healthy_groups': 9,
   'parkinson_groups': 10},
  'validation': {'images': 42,
   'participant_groups': 5,
   'healthy_images': 18,
   'parkinson_images': 24,
   'healthy_groups': 3,
   'parkinson_groups': 2},
  'test': {'images': 48,
   'participant_groups': 4,
   'healthy_images': 30,
   'parkinson_images': 18,
   'healthy_groups': 1,
   'parkinson_groups': 3}},
 'groupe

In [4]:
# Confirm the expected artifacts were written.
print('Model:', MODEL_PATH, MODEL_PATH.exists())
print('Results:', RESULTS_DIR)
sorted(p.name for p in RESULTS_DIR.iterdir())

Model: c:\Users\daisy\Projects\AI\pd-ml-lab\models\finalization_baseline\drawing_image_model.joblib True
Results: c:\Users\daisy\Projects\AI\pd-ml-lab\results\finalization_baseline\drawing


['drawing_candidate_results.csv', 'drawing_training_metrics.json']